In [ ]:
# input
# outperforms MIB2, Metal1D, BioMetAll
probe_dir = "./data/metal3d/probe/"
maxp_metal3d_file = "./data/metal3d/maxp_metal3d.csv"
metalnet_test_file = "../../../data/dataset/test_metalnet.tsv"

In [2]:
from Bio.PDB.MMCIFParser import MMCIFParser
import pandas as pd
from pathlib import Path

In [3]:

df_metalnet_test = pd.read_table(metalnet_test_file)
df_metalnet_test['resi_pdb_seq_num'] = df_metalnet_test['resi_pdb_id'].map(lambda x: int(x.split(",")[1]))
df_zn_truth = df_metalnet_test[df_metalnet_test['metal_resi'] == "ZN"]

In [4]:
from Bio.PDB.Residue import Residue
from Bio.PDB.Atom import Atom
import numpy as np

common_amino_acids = {
    'Ala', 'Cys', 'Asp', 'Glu', 'Phe',
    'Gly', 'His', 'Ile', 'Lys', 'Leu',
    'Met', 'Asn', 'Pro', 'Gln', 'Arg',
    'Ser', 'Thr', 'Val', 'Trp', 'Tyr',
}
hetero_atoms = {'N', 'O', 'S'}

def get_binding_residues(
    cif_file: Path,
    coords: list[np.ndarray],
    probas: list[float],
    distance_threshold: float = 3.0,
):
    id = cif_file.name.strip(".cif")
    struct = MMCIFParser(QUIET=True).get_structure(id, cif_file)
    m = next(struct.get_models())

    atoms = []
    for r in m.get_residues():
        r: Residue
        hetflag: str = r.get_id()[0]
        if hetflag == " " and str.capitalize(r.get_resname()) in common_amino_acids:
            ## these atoms are legal coordinate atoms
            atoms.extend([a for a in r.get_atoms() if a.element in hetero_atoms])
    
    records = []
    for index, coord in enumerate(coords):
        proba = probas[index]
        for a in atoms:
            a: Atom
            d = np.linalg.norm(a.get_coord() - coord)
            r = a.get_parent()
            if d < distance_threshold:
                record = {
                    "seq_id": id,
                    "resi_pdb_seq_num": r.get_id()[1],
                    "resi": str.capitalize(r.get_resname()),
                    "atom": a.get_name(),
                    "distance": d,
                    "metal_resi_proba": proba
                }
                records.append(record)
    return records

def get_coords_from_probe_file(
    probe_file: str
):
    coords = []
    probas = []
    with open(probe_file, "r") as f:
        lines = f.readlines()
        for l in lines:
            line = l.strip()
            coords.append(np.array([float(line[29:37]), float(line[37:45]), float(line[45:53])]))
            probas.append(float(line[55:59]))
    return coords, probas

In [5]:
import tqdm

records = []
df = pd.read_csv(maxp_metal3d_file)
for _, row in tqdm.tqdm(df.iterrows(), total=len(df)):
    pdb_file = Path(row['pdb'])
    probe_file = Path(probe_dir) / f"{pdb_file.name}.pdb"

    if probe_file.exists():
        coords, probas = get_coords_from_probe_file(probe_file)
    coords.append(np.array([row.x, row.y, row.z]))
    probas.append(1)

    records.extend(get_binding_residues(pdb_file, coords, probas))
df = pd.DataFrame(records)

  0%|          | 0/446 [00:00<?, ?it/s]

100%|██████████| 446/446 [00:40<00:00, 11.03it/s]


In [6]:
def calc_metrics_for_zn(df_pred):

    t_resi = set(zip(df_zn_truth["seq_id"], df_zn_truth["resi_pdb_seq_num"]))
    p_resi = set(zip(df_pred['seq_id'], df_pred['resi_pdb_seq_num']))
    inter_resi = t_resi & p_resi

    prec = len(inter_resi) / len(p_resi)
    recall = len(inter_resi) / len(t_resi)
    f1 = 2 * prec * recall / (prec + recall)

    return {
        "true": len(t_resi),
        "pred": len(p_resi),
        "inter": len(inter_resi),
        "precision": prec,
        "recall": recall,
        "f1": f1
    }

In [7]:
df = df.drop_duplicates(["seq_id","resi_pdb_seq_num"])
records = []
for i in range(0, 110, 10):
    cutoff = i / 100
    df_pred = df[df['metal_resi_proba'] >= cutoff]
    metrics = calc_metrics_for_zn(df_pred)
    metrics['proba_cutoff'] = cutoff
    records.append(metrics)

In [8]:
pd.DataFrame(records)

,true,pred,inter,precision,recall,f1,proba_cutoff
0,854,5386,827,0.153546,0.968384,0.265064,0.0
1,854,5386,827,0.153546,0.968384,0.265064,0.1
2,854,3803,824,0.216671,0.964871,0.353876,0.2
3,854,3149,820,0.260400,0.960187,0.409693,0.3
4,854,2787,816,0.292788,0.955504,0.448229,0.4
5,854,2450,812,0.331429,0.950820,0.491525,0.5
6,854,2114,805,0.380795,0.942623,0.542453,0.6
7,854,1789,791,0.442146,0.926230,0.598562,0.7
8,854,1532,772,0.503916,0.903981,0.647108,0.8
9,854,1261,744,0.590008,0.871194,0.703546,0.9
